In [1]:
import numpy as np
import pandas as pd

f = lambda x: x**2 - 6*x + 13

# ---------- Tasks 1-3 ----------
x0 = 1
print("Task 1:  f(x=1) =", f(1))                       # 1 - 6 + 13 = 8
xp = 2
df = f(xp) - f(x0)
print(f"Task 2:  neighbour x'=2, f(2)={f(2)},  Δf = f(2)-f(1) = {df}")
print("Task 3:  Δf ≤ 0  ->  P = 1  ->  x'=2 is ACCEPTED (better solution, no random number consumed)")

Task 1:  f(x=1) = 8
Task 2:  neighbour x'=2, f(2)=5,  Δf = f(2)-f(1) = -3
Task 3:  Δf ≤ 0  ->  P = 1  ->  x'=2 is ACCEPTED (better solution, no random number consumed)


In [2]:
def simulated_annealing(f, x0, T0, alpha, Tmin, randoms, n_iter=4):
    x, T = x0, T0
    best_x, best_f = x0, f(x0)
    r_iter = iter(randoms)
    rows = []
    for it in range(1, n_iter + 1):
        xp  = x + 1                              # neighbour rule x' = x + 1
        fx, fxp = f(x), f(xp)
        df  = fxp - fx
        if df <= 0:
            P, r, accept = 1.0, None, True       # better -> always accept
        else:
            P = np.exp(-df / T)
            r = next(r_iter)                     # consume the next given random number
            accept = (r <= P)
        rows.append({"Iter": it, "T": T, "x": x, "x'": xp, "f(x)": fx, "f(x')": fxp,
                     "Δf": df, "P": round(P, 4), "r": r if r is not None else "—",
                     "Decision": "Accept" if accept else "Reject"})
        if accept:
            x = xp
            if fxp < best_f: best_x, best_f = xp, fxp
        rows[-1]["Best x (f)"] = f"{best_x} ({best_f})"
        T *= alpha                               # cool AFTER each iteration
        if T < Tmin:
            print(f"(after iteration {it}: T = {T} < Tmin = {Tmin} -> algorithm would stop)")
    return pd.DataFrame(rows), best_x, best_f

table, best_x, best_f = simulated_annealing(f, x0=1, T0=10, alpha=0.5, Tmin=1,
                                            randoms=[0.30, 0.80, 0.20, 0.90])
table

(after iteration 4: T = 0.625 < Tmin = 1 -> algorithm would stop)


,Iter,T,x,x',f(x),f(x'),Δf,P,r,Decision,Best x (f)
0,1,10.00,1,2,8,5,-3,1.0000,—,Accept,2 (5)
1,2,5.00,2,3,5,4,-1,1.0000,—,Accept,3 (4)
2,3,2.50,3,4,4,5,1,0.6703,0.3,Accept,3 (4)
3,4,1.25,4,5,5,8,3,0.0907,0.8,Reject,3 (4)


In [3]:
# ---------- Tasks 6-7 ----------
print(f"Task 6:  best solution found by SA:  x* = {best_x},  f(x*) = {best_f}")
print()
print("Task 7 (analytical verification):")
print("  f'(x) = 2x - 6 = 0  ->  x = 3;   f''(x) = 2 > 0  ->  minimum")
print(f"  f(3) = 9 - 18 + 13 = {f(3)}   ->  the SA best solution coincides with the true minimum.")

Task 6:  best solution found by SA:  x* = 3,  f(x*) = 4

Task 7 (analytical verification):
  f'(x) = 2x - 6 = 0  ->  x = 3;   f''(x) = 2 > 0  ->  minimum
  f(3) = 9 - 18 + 13 = 4   ->  the SA best solution coincides with the true minimum.


In [4]:
def decode(s): return int(s, 2)

def ga_generation(pop, fit, pairs, points, title):
    """pop: list of bitstrings; fit: fitness fn on integer x;
       pairs: [(i,j),...] indices into the mating pool; points: crossover point per pair."""
    x  = np.array([decode(s) for s in pop])
    fx = np.array([fit(v) for v in x], float)
    tot, avg = fx.sum(), fx.mean()
    prob, exp_cnt = fx/tot, fx/avg
    # actual counts: round expected counts, then fix the total at len(pop)
    cnt = np.round(exp_cnt).astype(int)
    while cnt.sum() > len(pop): cnt[np.argmax(cnt - exp_cnt)] -= 1  # trim the most over-represented
    while cnt.sum() < len(pop): cnt[np.argmax(exp_cnt - cnt)] += 1

    t1 = pd.DataFrame({"String": pop, "x": x, "f(x)=x²": fx.astype(int) if fit is SQ else fx.round(4),
                       "Prob f/Σf": prob.round(4), "Expected f/avg(f)": exp_cnt.round(3),
                       "Actual count": cnt})
    print(f"==== {title} ====")
    print(t1.to_string(index=False))
    print(f"Σf = {tot:g}   max = {fx.max():g}   average = {avg:g}")

    pool = [s for s, c in zip(pop, cnt) for _ in range(c)]
    print("Mating pool:", pool)

    children = []
    for (i, j), p in zip(pairs, points):
        a, b = pool[i], pool[j]
        c1, c2 = a[:p] + b[p:], b[:p] + a[p:]
        print(f"  crossover point {p}:  {a} × {b}  ->  {c1}, {c2}")
        children += [c1, c2]
    cx  = np.array([decode(s) for s in children])
    cfx = np.array([fit(v) for v in cx], float)
    t2 = pd.DataFrame({"Offspring": children, "x": cx,
                       "fitness": cfx.astype(int) if fit is SQ else cfx.round(4)})
    print(t2.to_string(index=False))
    print(f"new Σf = {cfx.sum():g}   new max = {cfx.max():g}   new average = {cfx.mean():g}")
    print(f"improvement:  max {fx.max():g} -> {cfx.max():g},  average {avg:g} -> {cfx.mean():g}")
    print()
    return children

SQ = lambda v: v**2

In [5]:
gen2 = ga_generation(
    pop=["01100", "11001", "00101", "10011"], fit=SQ,
    pairs=[(0,1), (2,3)],          # (01100 × 11001) and (11001 × 10011)
    points=[4, 2],                 # the points that reproduce the notes' offspring
    title="GA Q1 — exercise population, Generation 1 -> 2")

==== GA Q1 — exercise population, Generation 1 -> 2 ====
String  x  f(x)=x²  Prob f/Σf  Expected f/avg(f)  Actual count
 01100 12      144     0.1247              0.499             1
 11001 25      625     0.5411              2.165             2
 00101  5       25     0.0216              0.087             0
 10011 19      361     0.3126              1.250             1
Σf = 1155   max = 625   average = 288.75
Mating pool: ['01100', '11001', '11001', '10011']
  crossover point 4:  01100 × 11001  ->  01101, 11000
  crossover point 2:  11001 × 10011  ->  11011, 10001
Offspring  x  fitness
    01101 13      169
    11000 24      576
    11011 27      729
    10001 17      289
new Σf = 1763   new max = 729   new average = 440.75
improvement:  max 625 -> 729,  average 288.75 -> 440.75



In [6]:
_ = ga_generation(
    pop=["11011", "10001", "01111", "10111"], fit=SQ,
    pairs=[(0,3), (1,2)],          # (11011 × 10111) and (11011 × 10001)
    points=[2, 2],
    title="GA Q1 — lecture example population, Generation 1 -> 2")

==== GA Q1 — lecture example population, Generation 1 -> 2 ====
String  x  f(x)=x²  Prob f/Σf  Expected f/avg(f)  Actual count
 11011 27      729     0.4114              1.646             2
 10001 17      289     0.1631              0.652             1
 01111 15      225     0.1270              0.508             0
 10111 23      529     0.2985              1.194             1
Σf = 1772   max = 729   average = 443
Mating pool: ['11011', '11011', '10001', '10111']
  crossover point 2:  11011 × 10111  ->  11111, 10011
  crossover point 2:  11011 × 10001  ->  11001, 10011
Offspring  x  fitness
    11111 31      961
    10011 19      361
    11001 25      625
    10011 19      361
new Σf = 2308   new max = 961   new average = 577
improvement:  max 729 -> 961,  average 443 -> 577



In [7]:
# ---- mutation step (as in the notes: bit-flip on a weak string) ----
def mutate(s, pos):                # pos is 0-indexed from the left
    return s[:pos] + ('1' if s[pos]=='0' else '0') + s[pos+1:]

ex = "00011"
print(f"mutation example from the notes:  {ex} (x={decode(ex)}, f={decode(ex)**2})"
      f"  -> flip bit 1 ->  {mutate(ex,0)} (x={decode(mutate(ex,0))}, f={decode(mutate(ex,0))**2})")
print()
print("role of mutation: a low-fitness string trapped near 00000 cannot reach the optimum")
print("11111 by crossover alone once diversity is lost; flipping a high-order bit re-injects it.")

mutation example from the notes:  00011 (x=3, f=9)  -> flip bit 1 ->  10011 (x=19, f=361)

role of mutation: a low-fitness string trapped near 00000 cannot reach the optimum
11111 by crossover alone once diversity is lost; flipping a high-order bit re-injects it.


In [8]:
gaq2_pop = ["11010", "00111", "10110", "00101"]
lo, hi, L = 0.0, 2.0, 5
fit2 = lambda x: -x**2 + 2*x

d  = np.array([decode(s) for s in gaq2_pop])
x  = lo + (hi - lo) / (2**L - 1) * d
fx = fit2(x)
tot = fx.sum()
prob = fx / tot
cum  = np.cumsum(prob)

t = pd.DataFrame({"String": gaq2_pop, "decimal": d, "x = 2d/31": x.round(4),
                  "f(x) = -x²+2x": fx.round(4), "prob": prob.round(4), "cumulative": cum.round(4)})
print(t.to_string(index=False))
print(f"Σf = {tot:.4f}   max f = {fx.max():.4f} at x = {x[np.argmax(fx)]:.4f}")

String  decimal  x = 2d/31  f(x) = -x²+2x   prob  cumulative
 11010       26     1.6774         0.5411 0.2077      0.2077
 00111        7     0.4516         0.6993 0.2684      0.4760
 10110       22     1.4194         0.8241 0.3163      0.7923
 00101        5     0.3226         0.5411 0.2077      1.0000
Σf = 2.6056   max f = 0.8241 at x = 1.4194


In [9]:
# ---- roulette-wheel selection with the GIVEN random numbers ----
randoms = [0.4, 0.15, 0.7, 0.9]
pool_idx = [int(np.searchsorted(cum, r)) for r in randoms]
pool = [gaq2_pop[i] for i in pool_idx]
for r, i in zip(randoms, pool_idx):
    print(f"r = {r:<5} falls in cumulative slot of string {i+1} ({gaq2_pop[i]})")
print("Mating pool:", pool)

r = 0.4   falls in cumulative slot of string 2 (00111)
r = 0.15  falls in cumulative slot of string 1 (11010)
r = 0.7   falls in cumulative slot of string 3 (10110)
r = 0.9   falls in cumulative slot of string 4 (00101)
Mating pool: ['00111', '11010', '10110', '00101']


In [10]:
# ---- crossover at the stated points: pair 1 at point 1, pair 2 at point 4 ----
pairs, points = [(0,1), (2,3)], [1, 4]
children = []
for (i,j), p in zip(pairs, points):
    a, b = pool[i], pool[j]
    c1, c2 = a[:p]+b[p:], b[:p]+a[p:]
    print(f"point {p}:  {a} × {b}  ->  {c1}, {c2}")
    children += [c1, c2]

dc  = np.array([decode(s) for s in children])
xc  = lo + (hi - lo)/(2**L - 1) * dc
fc  = fit2(xc)
t2 = pd.DataFrame({"Offspring": children, "decimal": dc, "x": xc.round(4), "f(x)": fc.round(4)})
print()
print(t2.to_string(index=False))
print(f"new Σf = {fc.sum():.4f}   new max = {fc.max():.4f} at x = {xc[np.argmax(fc)]:.4f}")
print(f"analytic optimum:  f'(x) = -2x + 2 = 0  ->  x* = 1,  f(x*) = 1")

point 1:  00111 × 11010  ->  01010, 10111
point 4:  10110 × 00101  ->  10111, 00100

Offspring  decimal      x   f(x)
    01010       10 0.6452 0.8741
    10111       23 1.4839 0.7659
    10111       23 1.4839 0.7659
    00100        4 0.2581 0.4495
new Σf = 2.8554   new max = 0.8741 at x = 0.6452
analytic optimum:  f'(x) = -2x + 2 = 0  ->  x* = 1,  f(x*) = 1
